# Improved Training Pipeline
This notebook loads `features.csv`, performs preprocessing, handles scaling, splits data properly, and trains multiple models with hyperparameter tuning to achieve strong performance.


In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

import warnings
warnings.filterwarnings("ignore")


In [ ]:

# Load dataset
data = pd.read_csv("features.csv")

print("Shape:", data.shape)
data.head()


In [ ]:

# Drop duplicates
data = data.drop_duplicates()

# Handle missing values
data = data.fillna(data.median(numeric_only=True))

print("Cleaned Shape:", data.shape)


In [ ]:

# Assume last column is target
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

print("Features shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


In [ ]:

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(random_state=42))
])

rf_params = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=5, n_jobs=-1)
rf_grid.fit(X_train, y_train)

rf_best = rf_grid.best_estimator_

rf_preds = rf_best.predict(X_test)

print("Random Forest Best Params:", rf_grid.best_params_)
print("Random Forest Accuracy:", accuracy_score(y_test, rf_preds))
print(classification_report(y_test, rf_preds))


In [ ]:

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC())
])

svm_params = {
    "clf__C": [0.1, 1, 10],
    "clf__kernel": ["linear", "rbf"],
    "clf__gamma": ["scale", "auto"]
}

svm_grid = GridSearchCV(svm_pipeline, svm_params, cv=5, n_jobs=-1)
svm_grid.fit(X_train, y_train)

svm_best = svm_grid.best_estimator_
svm_preds = svm_best.predict(X_test)

print("SVM Best Params:", svm_grid.best_params_)
print("SVM Accuracy:", accuracy_score(y_test, svm_preds))
print(classification_report(y_test, svm_preds))


In [ ]:

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_preds))
print(classification_report(y_test, lr_preds))


In [ ]:

models = {
    "RandomForest": accuracy_score(y_test, rf_preds),
    "SVM": accuracy_score(y_test, svm_preds),
    "LogisticRegression": accuracy_score(y_test, lr_preds)
}

best_model_name = max(models, key=models.get)
print("Best Model:", best_model_name)
print("Best Accuracy:", models[best_model_name])
